# Phase 4 – Python: Advanced Hotel Revenue Analysis

This notebook uses the `Raw_Data` sheet exported as `Hotel_Revenue_Raw_Data_For_PowerBI.csv`.
It covers data loading, cleaning, validation, KPIs, trends, segmentation, cancellation analysis, outliers, correlations, charts, and export files for your portfolio.

**Important:** This is a sample dataset with 30 bookings covering January–February 2025. Describe findings as sample-data findings, not conclusions about the whole hotel industry.

## Step 1 – Install and import libraries
Run the installation command once if the libraries are not already installed.

In [ ]:
%pip install pandas numpy matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

## Step 2 – Load your CSV file
Place the CSV in the same folder as this notebook. If it is elsewhere, replace the file path.

In [ ]:
file_path = 'Hotel_Revenue_Raw_Data_For_PowerBI.csv'
df = pd.read_csv(file_path)
df.head()

## Step 3 – Understand the dataset
Check the number of rows and columns, column names, data types, and basic statistics.

In [ ]:
print('Rows and columns:', df.shape)
print('\nColumn names:')
print(df.columns.tolist())
print('\nData types and non-null counts:')
df.info()
print('\nNumerical summary:')
display(df.describe(include='all').T)

## Step 4 – Clean and prepare the data
Convert date columns, remove accidental duplicate booking IDs only after checking them, and standardize text categories. Do not silently delete records.

In [ ]:
df['Booking_Date'] = pd.to_datetime(df['Booking_Date'], errors='coerce')
df['Check_In'] = pd.to_datetime(df['Check_In'], errors='coerce')
for col in ['Room_Type', 'Channel', 'Segment', 'Status', 'Weekend_Type']:
    df[col] = df[col].astype('string').str.strip()

print('Duplicate booking IDs:', df['Booking_ID'].duplicated().sum())
print('Missing values by column:')
display(df.isna().sum().to_frame('Missing_Count'))
print('Status values:', df['Status'].value_counts(dropna=False))
print('Date conversion complete.')

## Step 5 – Validate business rules
Check positive nights and guest counts, non-negative revenue, valid status values, and whether cancellation flags agree with status. Review any flagged rows before changing them.

In [ ]:
checks = {
    'Duplicate Booking_ID': df['Booking_ID'].duplicated().sum(),
    'Missing Booking_ID': df['Booking_ID'].isna().sum(),
    'Missing booking/check-in dates': df[['Booking_Date','Check_In']].isna().any(axis=1).sum(),
    'Nights <= 0': (df['Nights'] <= 0).sum(),
    'Guests <= 0': (df['Guests'] <= 0).sum(),
    'Negative Actual_Revenue': (df['Actual_Revenue'] < 0).sum(),
    'Unknown Status': (~df['Status'].isin(['Stayed','Cancelled'])).sum(),
    'Cancellation flag mismatch': (df['Cancellation_Flag'] != (df['Status'] == 'Cancelled').astype(int)).sum()
}
display(pd.Series(checks, name='Issue_Count').to_frame())

## Step 6 – Create the main hotel KPIs
Use `Actual_Revenue` for realized revenue. Calculate realized ADR as total realized revenue divided by occupied room nights. This is a weighted ADR and is more appropriate than simply averaging each booking's ADR.

In [ ]:
total_bookings = len(df)
stayed = df[df['Status'] == 'Stayed'].copy()
cancelled = df[df['Status'] == 'Cancelled'].copy()
total_revenue = df['Actual_Revenue'].sum()
occupied_room_nights = df['Occupied_Room_Nights'].sum()
realized_adr = total_revenue / occupied_room_nights if occupied_room_nights else np.nan
cancellation_rate = len(cancelled) / total_bookings * 100 if total_bookings else 0

kpis = pd.DataFrame({
    'Metric': ['Total Bookings','Stayed Bookings','Cancelled Bookings','Total Revenue','Occupied Room Nights','Realized ADR','Cancellation Rate'],
    'Value': [total_bookings, len(stayed), len(cancelled), total_revenue, occupied_room_nights, realized_adr, cancellation_rate]
})
display(kpis)

## Step 7 – Monthly revenue and booking trends
Group by month and compare revenue, booking count, stayed bookings, and cancellations.

In [ ]:
monthly = df.groupby('Month', dropna=False).agg(
    Bookings=('Booking_ID','count'),
    Revenue=('Actual_Revenue','sum'),
    Stayed_Bookings=('Status', lambda s: (s == 'Stayed').sum()),
    Cancellations=('Cancellation_Flag','sum'),
    Occupied_Room_Nights=('Occupied_Room_Nights','sum')
).reset_index()
monthly['Realized_ADR'] = monthly['Revenue'] / monthly['Occupied_Room_Nights'].replace(0, np.nan)
display(monthly)

plt.figure(figsize=(8,4))
plt.bar(monthly['Month'].astype(str), monthly['Revenue'])
plt.title('Monthly Hotel Revenue')
plt.xlabel('Month'); plt.ylabel('Revenue ($)'); plt.tight_layout(); plt.show()

## Step 8 – Revenue by room type and booking channel
Find which room types and channels contribute the most realized revenue.

In [ ]:
room_analysis = df.groupby('Room_Type').agg(
    Bookings=('Booking_ID','count'), Revenue=('Actual_Revenue','sum'),
    Occupied_Nights=('Occupied_Room_Nights','sum')
).reset_index().sort_values('Revenue', ascending=False)
room_analysis['Realized_ADR'] = room_analysis['Revenue'] / room_analysis['Occupied_Nights'].replace(0, np.nan)
display(room_analysis)

channel_analysis = df.groupby('Channel').agg(
    Bookings=('Booking_ID','count'), Revenue=('Actual_Revenue','sum'),
    Cancellations=('Cancellation_Flag','sum')
).reset_index().sort_values('Revenue', ascending=False)
display(channel_analysis)

plt.figure(figsize=(8,4))
plt.bar(room_analysis['Room_Type'], room_analysis['Revenue'])
plt.title('Revenue by Room Type'); plt.xlabel('Room Type'); plt.ylabel('Revenue ($)'); plt.tight_layout(); plt.show()

## Step 9 – Customer segment and weekday/weekend analysis
Compare business and leisure guests, and weekday versus weekend performance.

In [ ]:
segment_analysis = df.groupby('Segment').agg(
    Bookings=('Booking_ID','count'), Revenue=('Actual_Revenue','sum'),
    Occupied_Nights=('Occupied_Room_Nights','sum')
).reset_index()
display(segment_analysis)

weekend_analysis = df.groupby('Weekend_Type').agg(
    Bookings=('Booking_ID','count'), Revenue=('Actual_Revenue','sum'),
    Occupied_Nights=('Occupied_Room_Nights','sum')
).reset_index()
display(weekend_analysis)

## Step 10 – Cancellation analysis
Measure cancellation rate by channel, room type, and customer segment. Use booking counts as the denominator.

In [ ]:
def cancellation_summary(group_col):
    result = df.groupby(group_col).agg(
        Bookings=('Booking_ID','count'), Cancellations=('Cancellation_Flag','sum')
    ).reset_index()
    result['Cancellation_Rate_Pct'] = result['Cancellations'] / result['Bookings'] * 100
    return result.sort_values('Cancellation_Rate_Pct', ascending=False)

display(cancellation_summary('Channel'))
display(cancellation_summary('Room_Type'))
display(cancellation_summary('Segment'))

## Step 11 – Lead-time and length-of-stay analysis
Explore how far in advance guests book and how long they stay. These are descriptive patterns, not proof of cause and effect.

In [ ]:
df['Lead_Time_Group'] = pd.cut(
    df['Lead_Time_Days'], bins=[-1,7,30,60,np.inf],
    labels=['0–7 days','8–30 days','31–60 days','61+ days']
)
lead_analysis = df.groupby('Lead_Time_Group', observed=False).agg(
    Bookings=('Booking_ID','count'), Revenue=('Actual_Revenue','sum'),
    Cancellations=('Cancellation_Flag','sum')
).reset_index()
display(lead_analysis)

stay_analysis = df.groupby('Nights').agg(
    Bookings=('Booking_ID','count'), Revenue=('Actual_Revenue','sum')
).reset_index().sort_values('Nights')
display(stay_analysis)

## Step 12 – Detect unusual values (outlier review)
Use the IQR method to flag unusually high or low ADR and booking revenue. An outlier is a review flag, not automatically an error.

In [ ]:
def iqr_outlier_report(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return lower, upper

for col in ['ADR','Actual_Revenue']:
    low, high = iqr_outlier_report(df[col].dropna())
    flagged = df[(df[col] < low) | (df[col] > high)][['Booking_ID',col,'Room_Type','Channel','Status']]
    print(f'{col}: lower threshold={low:.2f}, upper threshold={high:.2f}')
    display(flagged)

## Step 13 – Correlation analysis
Check numerical relationships. Correlation does not prove that one variable causes another.

In [ ]:
numeric_cols = ['Nights','Guests','Lead_Time_Days','ADR','Actual_Revenue','Occupied_Room_Nights']
correlations = df[numeric_cols].corr(numeric_only=True)
display(correlations)

plt.figure(figsize=(8,6))
plt.imshow(correlations, aspect='auto')
plt.xticks(range(len(numeric_cols)), numeric_cols, rotation=45, ha='right')
plt.yticks(range(len(numeric_cols)), numeric_cols)
plt.colorbar(label='Correlation')
plt.title('Correlation Matrix')
plt.tight_layout(); plt.show()

## Step 14 – Write business findings
After running the notebook, write 3–5 findings supported by the output. Example format:
- The room type with the highest realized revenue was ____.
- Revenue in February was ____ compared with January.
- The booking channel with the highest cancellation rate was ____.
- The lead-time group with the most bookings was ____.
- The outlier review identified ____ records that should be checked.

Do not claim that a pattern caused another result unless you have evidence for causation.

## Step 15 – Export analysis tables for Power BI and your portfolio
These CSV files can be used as supporting outputs. Your main Power BI dashboard can still use the cleaned booking-level data.

In [ ]:
output_dir = Path('python_analysis_outputs')
output_dir.mkdir(exist_ok=True)

df.to_csv(output_dir / 'hotel_cleaned_data.csv', index=False)
kpis.to_csv(output_dir / 'hotel_kpis.csv', index=False)
monthly.to_csv(output_dir / 'monthly_performance.csv', index=False)
room_analysis.to_csv(output_dir / 'room_type_analysis.csv', index=False)
channel_analysis.to_csv(output_dir / 'channel_analysis.csv', index=False)
segment_analysis.to_csv(output_dir / 'segment_analysis.csv', index=False)
weekend_analysis.to_csv(output_dir / 'weekday_weekend_analysis.csv', index=False)
lead_analysis.to_csv(output_dir / 'lead_time_analysis.csv', index=False)

print('Export complete. Files saved in:', output_dir.resolve())

## Step 16 – Portfolio checklist
- [ ] Save this notebook as `Hotel_Revenue_Advanced_Analysis.ipynb`.
- [ ] Run every cell from top to bottom.
- [ ] Review and explain every flagged data-quality issue or outlier.
- [ ] Save the exported CSV tables.
- [ ] Add 3–5 evidence-based business findings to your project report.
- [ ] Publish the notebook and supporting outputs to GitHub.
- [ ] Use the cleaned data or exported summary tables in Power BI.

**Portfolio note:** Keep your original raw data unchanged. Document any cleaning decisions so another analyst can reproduce your work.